# 07 Pairwise Similarity: gpt-4o vs gpt-5.1

Uses `data/comparison-pairs.jsonl` — five hand-crafted question/answer pairs, one response per model per row.

`SimilarityEvaluator` is called with:
- `response` = gpt-5.1 answer
- `ground_truth` = gpt-4o answer

A score of 1 means the answers are very different; 5 means they are nearly identical in wording. The evaluator model is gpt-4o (`chat4o`).

## Load environment

In [ ]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path('../env/.env')
if not env_path.exists():
    raise RuntimeError('env/.env not found. Run 01_deploy_infra.ipynb first.')

load_dotenv(env_path)

AOAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', '').rstrip('/')
AOAI_KEY = os.getenv('AZURE_OPENAI_API_KEY', '')
AOAI_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'chat4o')
AOAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21')

print('Environment loaded.')
print(f'  endpoint   : {AOAI_ENDPOINT}')
print(f'  deployment : {AOAI_DEPLOYMENT}')
print(f'  api_version: {AOAI_API_VERSION}')

## Run SimilarityEvaluator

In [ ]:
from azure.ai.evaluation import SimilarityEvaluator, RelevanceEvaluator

pairs_path = Path('../data/comparison-pairs.jsonl')
# utf-8-sig handles BOM that Windows tools may add
pairs = [json.loads(line) for line in pairs_path.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
print(f'Loaded {len(pairs)} comparison pairs')

model_config = {
    'azure_endpoint': AOAI_ENDPOINT,
    'api_key': AOAI_KEY,
    'azure_deployment': AOAI_DEPLOYMENT,
    'api_version': '2024-08-01-preview',
}

similarity_evaluator = SimilarityEvaluator(model_config=model_config)
relevance_evaluator = RelevanceEvaluator(model_config=model_config)

similarity_rows = []

for i, pair in enumerate(pairs, start=1):
    query = pair['query']
    response_4o = pair['response_4o']
    response_51 = pair['response_51']

    # Similarity score: gpt-4o answer as ground truth, gpt-5.1 answer as prediction
    sim_result = similarity_evaluator(
        query=query,
        response=response_51,
        ground_truth=response_4o,
    )
    similarity_score = sim_result.get('similarity', None)

    # Evaluator-native rationale
    rel_result = relevance_evaluator(
        query=query,
        response=response_51,
        context=response_4o,
    )
    evaluator_reason = rel_result.get('relevance_reason', '')

    similarity_rows.append({
        'pair': i,
        'query': query,
        'similarity': similarity_score,
        'evaluator_reason': evaluator_reason,
        'response_4o': response_4o,
        'response_51': response_51,
    })

    print(f"Pair {i}  similarity={similarity_score}  query={query[:60]!r}")

print(f'\nDone - {len(similarity_rows)} pairs scored.')

## Display results

In [ ]:
from IPython.display import Markdown, display
import importlib
import sys

app_dir = Path('../app').resolve()
if str(app_dir) not in sys.path:
    sys.path.append(str(app_dir))

import comparison_renderer
importlib.reload(comparison_renderer)

# Notebook 05-style full rendering: no truncation of model responses
render_rows = []
for row in similarity_rows:
    render_rows.append({
        'scenario_id': f"pair_{row['pair']:02d}",
        'category': 'Pairwise Similarity',
        'system_label': 'Similarity score',
        'system': f"{row['similarity']}",
        'user_label': 'Prompt',
        'user': row['query'],
        'difference_label': 'Evaluator reasoning',
        'difference_explanation': row.get('evaluator_reason', ''),
        'primary_deployment': AOAI_DEPLOYMENT,
        'secondary_deployment': 'chat51',
        'primary_status': 'success',
        'secondary_status': 'success',
        'primary_response': row['response_4o'],
        'secondary_response': row['response_51'],
    })

comparison_renderer.render_comparison(render_rows)

avg = sum(r['similarity'] for r in similarity_rows if isinstance(r['similarity'], (int, float))) / len(similarity_rows)
display(Markdown(f"Average similarity across {len(similarity_rows)} pairs: **{avg:.2f}**"))

# Save results
output_path = Path('../outputs/similarity-pairs-results.json')
output_path.write_text(json.dumps(similarity_rows, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Saved to {output_path}')